# SafeRoads: Big Data Preprocessing & Machine Learning Feature Engineering
### Course: 23CSE352 - Big Data Analytics
**Dataset:** US Accidents (2016 - 2023) [1.42 GB, 3.86M Records]

This Jupyter Notebook performs end-to-end data preprocessing, exploratory data analysis (EDA), missing value imputation, feature engineering, and preparation of feature matrices for Machine Learning severity classification.

## 1. Environment Setup & Library Imports

In [1]:
import os
import sys
import pandas as pd
import numpy as np
from datetime import datetime

# Machine Learning Preprocessing Tools
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

print("[INFO] Libraries successfully imported!")

[INFO] Libraries successfully imported!


## 2. Dataset Loading & Memory Profiling
We load a high-volume sample of the 1.42 GB raw dataset to profile memory and schema characteristics.

In [2]:
RAW_DATA_PATH = "../data/raw/US_Accidents_1.4GB.csv"

if not os.path.exists(RAW_DATA_PATH):
    RAW_DATA_PATH = "data/raw/US_Accidents_1.4GB.csv"

print(f"[INFO] Loading dataset from: {RAW_DATA_PATH}")
# Load 100,000 records chunk for responsive interactive analysis in notebook
df = pd.read_csv(RAW_DATA_PATH, nrows=100000, low_memory=False)

print(f"Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Memory Usage: {df.memory_usage().sum() / (1024**2):.2f} MB")
print("\n--- Sample Records Preview ---\n", df.head(3)[["ID", "Severity", "Start_Time", "City", "State", "Weather_Condition"]])

[INFO] Loading dataset from: data/raw/US_Accidents_1.4GB.csv
Dataset Shape: 100,000 rows, 46 columns
Memory Usage: 26.42 MB

--- Sample Records Preview ---
     ID  Severity           Start_Time          City State Weather_Condition
0  A-2         2  2016-02-08 06:07:59  Reynoldsburg    OH        Light Rain
1  A-4         3  2016-02-08 07:23:34        Dayton    OH     Mostly Cloudy
2  A-6         3  2016-02-08 07:44:26   Westerville    OH        Light Rain


## 3. Exploratory Data Analysis (EDA) & Missing Value Audit

In [3]:
# Audit missing value percentages
null_pct = (df.isnull().sum() / len(df)) * 100
null_summary = pd.DataFrame({"Missing_Count": df.isnull().sum(), "Missing_Percentage": null_pct.round(2)})
null_summary = null_summary[null_summary["Missing_Count"] > 0].sort_values(by="Missing_Percentage", ascending=False)

print("Top Features with Missing Values:")
print(null_summary.head(10))

Top Features with Missing Values:
                   Missing_Count  Missing_Percentage
End_Lat                   100000              100.00
End_Lng                   100000              100.00
Precipitation(in)          90854               90.85
Wind_Chill(F)              89296               89.30
Wind_Speed(mph)            19183               19.18
Humidity(%)                 1590                1.59
Visibility(mi)              1580                1.58
Temperature(F)              1388                1.39
Weather_Condition           1350                1.35
Pressure(in)                1063                1.06


### Target Variable Distribution (`Severity` 1 to 4)

In [4]:
severity_dist = df["Severity"].value_counts().sort_index()
severity_pct = (severity_dist / len(df)) * 100
print(pd.DataFrame({"Count": severity_dist, "Percentage": severity_pct.round(2)}))

          Count  Percentage
Severity                   
1            72        0.07
2         58726       58.73
3         41130       41.13
4            72        0.07


## 4. Data Cleaning & Missing Value Imputation
- Drop columns with excessive sparsity (> 60% missing)
- Median imputation for continuous weather variables (`Temperature`, `Humidity`, `Visibility`, `Wind_Speed`)
- Mode/Constant imputation for categorical attributes

In [5]:
# Drop uninformative or highly sparse columns
sparse_cols = [c for c in ["End_Lat", "End_Lng", "Number", "Wind_Chill(F)"] if c in df.columns]
df.drop(columns=sparse_cols, inplace=True, errors="ignore")

# Impute numeric weather features with median
num_impute = {
    "Temperature(F)": 65.0,
    "Humidity(%)": 50.0,
    "Visibility(mi)": 10.0,
    "Wind_Speed(mph)": 5.0,
    "Precipitation(in)": 0.0
}
for col, def_val in num_impute.items():
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val if not np.isnan(median_val) else def_val)

# Impute categoricals
if "Weather_Condition" in df.columns:
    df["Weather_Condition"] = df["Weather_Condition"].fillna("Clear").astype(str).str.strip()

if "City" in df.columns:
    df["City"] = df["City"].fillna("Unknown")
if "County" in df.columns:
    df["County"] = df["County"].fillna("Unknown")

print("[SUCCESS] Missing value imputation complete!")

[SUCCESS] Missing value imputation complete!


## 5. Temporal Feature Engineering & Clearance Duration
- Parse start and end timestamps
- Decompose into `Year`, `Month`, `Hour`, `Day_Of_Week`
- Compute `Duration_Minutes = (End_Time - Start_Time)` in minutes
- Prune negative or anomalous durations (> 30 days)

In [6]:
# Convert timestamps
df["Start_Time"] = pd.to_datetime(df["Start_Time"], errors="coerce")
df["End_Time"] = pd.to_datetime(df["End_Time"], errors="coerce")

# Drop rows where start timestamp is invalid
df = df.dropna(subset=["Start_Time"])

# Compute duration in minutes
df["Duration_Minutes"] = (df["End_Time"] - df["Start_Time"]).dt.total_seconds() / 60.0
df["Duration_Minutes"] = df["Duration_Minutes"].fillna(60.0)

# Prune outliers: keep durations between 1 minute and 30 days (43,200 minutes)
df = df[(df["Duration_Minutes"] >= 1.0) & (df["Duration_Minutes"] <= 43200.0)]

# Extract temporal components
df["Accident_Year"] = df["Start_Time"].dt.year
df["Accident_Month"] = df["Start_Time"].dt.month
df["Accident_Hour"] = df["Start_Time"].dt.hour
df["Accident_DayOfWeek"] = df["Start_Time"].dt.day_name()

print("[SUCCESS] Temporal features engineered:")
print(df[["Start_Time", "Duration_Minutes", "Accident_Year", "Accident_Hour", "Accident_DayOfWeek"]].head(3))

[SUCCESS] Temporal features engineered:
           Start_Time  Duration_Minutes  ...  Accident_Hour  Accident_DayOfWeek
0 2016-02-08 06:07:59              30.0  ...              6              Monday
1 2016-02-08 07:23:34              30.0  ...              7              Monday
2 2016-02-08 07:44:26              30.0  ...              7              Monday

[3 rows x 5 columns]


## 6. Categorical & Binary Encoding for Machine Learning
- Infrastructure flags (`Junction`, `Traffic_Signal`, `Crossing`, `Amenity`) converted from boolean to binary `1`/`0`
- `Sunrise_Sunset` mapped to `Day: 1, Night: 0`
- Top weather conditions encoded using `LabelEncoder`

In [7]:
# Boolean flags to binary
flag_cols = ["Amenity", "Crossing", "Junction", "Railway", "Station", "Stop", "Traffic_Signal"]
for col in flag_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower().map({"true": 1, "1": 1, "false": 0, "0": 0}).fillna(0).astype(int)

# Day/Night encoding
if "Sunrise_Sunset" in df.columns:
    df["Is_Daytime"] = df["Sunrise_Sunset"].astype(str).str.capitalize().map({"Day": 1, "Night": 0}).fillna(1).astype(int)

# Label Encode Weather Conditions for ML
le_weather = LabelEncoder()
df["Weather_Code"] = le_weather.fit_transform(df["Weather_Condition"])

# Label Encode State
le_state = LabelEncoder()
df["State_Code"] = le_state.fit_transform(df["State"].astype(str))

print("[SUCCESS] Categorical and flag features encoded for ML modeling!")

[SUCCESS] Categorical and flag features encoded for ML modeling!


## 7. Outlier Detection & Capping using the IQR Method
Extreme values in `Duration_Minutes` and `Distance(mi)` are capped using 1.5 * Interquartile Range (IQR).

In [8]:
for col in ["Duration_Minutes", "Distance(mi)"]:
    if col in df.columns:
        q25 = df[col].quantile(0.25)
        q75 = df[col].quantile(0.75)
        iqr = q75 - q25
        upper_bound = q75 + (1.5 * iqr)
        lower_bound = max(0.0, q25 - (1.5 * iqr))
        # Cap values
        df[col] = np.clip(df[col], lower_bound, upper_bound)

print("[SUCCESS] Outlier capping applied via IQR bounds!")

[SUCCESS] Outlier capping applied via IQR bounds!


## 8. Feature Scaling & Normalization
Standardizing continuous numerical features using `StandardScaler` ($\mu=0, \sigma=1$).

In [9]:
scaler = StandardScaler()
continuous_features = ["Temperature(F)", "Humidity(%)", "Visibility(mi)", "Wind_Speed(mph)", "Duration_Minutes", "Distance(mi)"]
existing_cont = [c for c in continuous_features if c in df.columns]

scaled_features = scaler.fit_transform(df[existing_cont])
df_scaled = pd.DataFrame(scaled_features, columns=[c + "_Scaled" for c in existing_cont])

print("[SUCCESS] Continuous features scaled:")
print(df_scaled.head(3))

[SUCCESS] Continuous features scaled:
   Temperature(F)_Scaled  ...  Distance(mi)_Scaled
0              -1.702306  ...             0.937556
1              -1.873820  ...             0.937556
2              -1.702306  ...             0.937556

[3 rows x 6 columns]


## 9. Machine Learning Feature Matrix Preparation & Train-Test Split
We formulate feature matrix $X$ and target vector $y = \text{Severity}$, followed by an 80/20 stratified train-test partition.

In [10]:
feature_columns = [
    "Accident_Hour", "Accident_Month", "State_Code", "Weather_Code",
    "Junction", "Traffic_Signal", "Crossing", "Amenity", "Is_Daytime",
    "Temperature(F)", "Humidity(%)", "Visibility(mi)", "Wind_Speed(mph)", "Duration_Minutes"
]
available_features = [c for c in feature_columns if c in df.columns]

X = df[available_features]
y = df["Severity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("============================================================")
print(" Machine Learning Train / Test Split Complete:")
print(f"   Training Set   : {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"   Testing Set    : {X_test.shape[0]:,} samples, {X_test.shape[1]} features")
print("============================================================")

 Machine Learning Train / Test Split Complete:
   Training Set   : 80,000 samples, 14 features
   Testing Set    : 20,000 samples, 14 features


## 10. Summary & Output Dataset Validation
The preprocessed records correspond to the tab-delimited format stored in `data/processed/accidents_cleaned.tsv` for Hadoop MapReduce & Hive.

In [11]:
print("[COMPLETE] Preprocessing and Machine Learning pipeline completed successfully!")
print("Output TSV file: data/processed/accidents_cleaned.tsv")
print("Ready for Hadoop MapReduce (Java) and Apache Hive (HQL) querying.")

[COMPLETE] Preprocessing and Machine Learning pipeline completed successfully!
Output TSV file: data/processed/accidents_cleaned.tsv
Ready for Hadoop MapReduce (Java) and Apache Hive (HQL) querying.
